# 🔥 Fraud Detection - Spark Streaming Pipeline

## Architecture
```
Event Hubs ──▶ Spark Streaming ──▶ Data Lake (Parquet) ──▶ Synapse SQL
```

### Ce notebook:
1. Lit les transactions depuis Azure Event Hubs
2. Enrichit les données avec des features
3. Applique les règles de détection de fraude
4. Sauvegarde vers Azure Data Lake Storage
5. Publie les alertes vers un autre Event Hub

---
## Configuration

In [ ]:
# Configuration - MODIFIER CES VALEURS
EVENT_HUB_CONNECTION_STRING = "Endpoint=sb://evh-fraud-detection-xxx.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=xxx"
EVENT_HUB_NAME = "transactions"
ALERT_EVENT_HUB_NAME = "fraud-alerts"

STORAGE_ACCOUNT = "stfrauddetectionxxx"  # Votre storage account
CONTAINER_PROCESSED = "processed-data"
CONTAINER_ALERTS = "fraud-alerts"

# Paths Data Lake
OUTPUT_PATH = f"abfss://{CONTAINER_PROCESSED}@{STORAGE_ACCOUNT}.dfs.core.windows.net/transactions/"
ALERTS_PATH = f"abfss://{CONTAINER_ALERTS}@{STORAGE_ACCOUNT}.dfs.core.windows.net/alerts/"
CHECKPOINT_PATH = f"abfss://{CONTAINER_PROCESSED}@{STORAGE_ACCOUNT}.dfs.core.windows.net/checkpoints/streaming/"

print(f"✅ Configuration:")
print(f"   Output: {OUTPUT_PATH}")
print(f"   Alerts: {ALERTS_PATH}")

---
## Imports et Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Spark Session (déjà disponible dans Synapse comme 'spark')
print(f"✅ Spark version: {spark.version}")
print(f"   Application: {spark.sparkContext.appName}")

---
## Schema des Transactions

In [ ]:
# Schéma des transactions JSON
transaction_schema = StructType([
    StructField("transactionId", StringType(), False),
    StructField("customerId", StringType(), False),
    StructField("merchantId", StringType(), False),
    StructField("amount", DoubleType(), False),
    StructField("currency", StringType(), True),
    StructField("channel", StringType(), True),
    StructField("location", StringType(), True),
    StructField("country", StringType(), True),
    StructField("timestamp", LongType(), False),
    StructField("timestampStr", StringType(), True),
    StructField("cardType", StringType(), True),
    StructField("isInternational", BooleanType(), True),
    StructField("merchantCategory", StringType(), True),
    StructField("previousBalance", DoubleType(), True),
    StructField("isFraud", BooleanType(), True),
    StructField("fraudType", StringType(), True)
])

print("✅ Schema défini")

---
## Lecture depuis Event Hubs (Streaming)

In [ ]:
# Configuration Event Hubs pour Spark
ehConf = {
    'eventhubs.connectionString': sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(EVENT_HUB_CONNECTION_STRING),
    'eventhubs.eventHubName': EVENT_HUB_NAME,
    'eventhubs.consumerGroup': '$Default',
    'eventhubs.startingPosition': '{"offset":"-1","seqNo":-1,"enqueuedTime":null,"isInclusive":true}'
}

# Lecture streaming depuis Event Hubs
raw_stream = spark.readStream \
    .format("eventhubs") \
    .options(**ehConf) \
    .load()

print("✅ Stream Event Hubs configuré")
raw_stream.printSchema()

---
## Parser les messages JSON

In [ ]:
# Parser le body JSON
parsed_stream = raw_stream \
    .select(
        F.col("enqueuedTime").alias("eventTime"),
        F.col("offset"),
        F.col("partitionId"),
        F.from_json(
            F.col("body").cast("string"), 
            transaction_schema
        ).alias("data")
    ) \
    .select("eventTime", "offset", "partitionId", "data.*")

print("✅ Parser configuré")
parsed_stream.printSchema()

---
## Enrichissement des données

In [ ]:
def enrich_transactions(df):
    """
    Enrichir les transactions avec des features calculées
    """
    return df \
        .withColumn(
            "processedAt", 
            F.current_timestamp()
        ) \
        .withColumn(
            "transactionTime",
            F.from_unixtime(F.col("timestamp") / 1000)
        ) \
        .withColumn(
            "hour",
            F.hour(F.from_unixtime(F.col("timestamp") / 1000))
        ) \
        .withColumn(
            "dayOfWeek",
            F.dayofweek(F.from_unixtime(F.col("timestamp") / 1000))
        ) \
        .withColumn(
            "isSuspiciousHour",
            F.col("hour").between(0, 5)
        ) \
        .withColumn(
            "isRiskyMerchant",
            F.col("merchantCategory").isin("GAMBLING", "CRYPTO")
        ) \
        .withColumn(
            "isHighAmount",
            F.col("amount") > 5000
        ) \
        .withColumn(
            "amountToBalanceRatio",
            F.when(
                F.col("previousBalance") > 0,
                F.col("amount") / F.col("previousBalance")
            ).otherwise(F.lit(1.0))
        ) \
        .withColumn(
            "amountCategory",
            F.when(F.col("amount") < 50, "MICRO")
            .when(F.col("amount") < 200, "LOW")
            .when(F.col("amount") < 1000, "MEDIUM")
            .when(F.col("amount") < 5000, "HIGH")
            .otherwise("VERY_HIGH")
        )

enriched_stream = enrich_transactions(parsed_stream)
print("✅ Enrichissement configuré")

---
## Règles de détection de fraude

In [ ]:
def apply_fraud_rules(df):
    """
    Appliquer les règles de scoring de fraude
    """
    return df \
        .withColumn(
            "fraudScore",
            # Score basé sur le montant
            F.when(F.col("amount") > 10000, F.lit(0.5))
            .when(F.col("amount") > 5000, F.lit(0.3))
            .when(F.col("amount") > 2000, F.lit(0.15))
            .otherwise(F.lit(0.0))
            # + Score marchand risqué
            + F.when(F.col("isRiskyMerchant"), F.lit(0.3)).otherwise(F.lit(0.0))
            # + Score heure suspecte
            + F.when(F.col("isSuspiciousHour"), F.lit(0.15)).otherwise(F.lit(0.0))
            # + Score international + montant élevé
            + F.when(
                F.col("isInternational") & (F.col("amount") > 1000),
                F.lit(0.25)
            ).otherwise(F.lit(0.0))
            # + Score ratio amount/balance
            + F.when(
                F.col("amountToBalanceRatio") > 0.8,
                F.lit(0.2)
            ).otherwise(F.lit(0.0))
        ) \
        .withColumn(
            "riskLevel",
            F.when(F.col("fraudScore") >= 0.7, "CRITICAL")
            .when(F.col("fraudScore") >= 0.5, "HIGH")
            .when(F.col("fraudScore") >= 0.3, "MEDIUM")
            .otherwise("LOW")
        ) \
        .withColumn(
            "isFraudPredicted",
            F.col("fraudScore") >= 0.5
        ) \
        .withColumn(
            "alertRequired",
            F.col("riskLevel").isin("HIGH", "CRITICAL")
        )

scored_stream = apply_fraud_rules(enriched_stream)
print("✅ Règles de fraude configurées")

---
## Colonnes finales pour partition

In [ ]:
# Ajouter colonnes de partition pour Data Lake
final_stream = scored_stream \
    .withColumn(
        "transaction_date",
        F.to_date(F.from_unixtime(F.col("timestamp") / 1000))
    ) \
    .withColumn(
        "processing_date",
        F.current_date()
    )

print("✅ Stream final prêt")
final_stream.printSchema()

---
## Écriture vers Data Lake (Parquet)

In [ ]:
# Écriture streaming vers Data Lake en Parquet partitionné
query_datalake = final_stream \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", OUTPUT_PATH) \
    .option("checkpointLocation", CHECKPOINT_PATH + "transactions/") \
    .partitionBy("riskLevel", "transaction_date") \
    .trigger(processingTime="30 seconds") \
    .start()

print(f"✅ Streaming vers Data Lake démarré")
print(f"   Path: {OUTPUT_PATH}")
print(f"   Query ID: {query_datalake.id}")

---
## Écriture des alertes

In [ ]:
# Filtrer les alertes (HIGH et CRITICAL)
alerts_stream = final_stream \
    .filter(F.col("alertRequired") == True) \
    .select(
        "transactionId",
        "customerId",
        "merchantId",
        "amount",
        "merchantCategory",
        "location",
        "fraudScore",
        "riskLevel",
        "processedAt",
        "transaction_date"
    )

# Écriture des alertes vers Data Lake
query_alerts = alerts_stream \
    .writeStream \
    .format("parquet") \
    .outputMode("append") \
    .option("path", ALERTS_PATH) \
    .option("checkpointLocation", CHECKPOINT_PATH + "alerts/") \
    .partitionBy("riskLevel", "transaction_date") \
    .trigger(processingTime="30 seconds") \
    .start()

print(f"✅ Streaming alertes démarré")
print(f"   Path: {ALERTS_PATH}")

---
## Monitoring du Stream

In [ ]:
# Afficher le status des queries
import time

def monitor_streams(duration_minutes=5):
    """
    Monitor les streams pendant X minutes
    """
    print("📊 Monitoring des streams...")
    print("="*60)
    
    end_time = time.time() + (duration_minutes * 60)
    
    while time.time() < end_time:
        for query in spark.streams.active:
            status = query.status
            progress = query.lastProgress
            
            print(f"\n🔄 Query: {query.name or query.id[:8]}")
            print(f"   Status: {'Running ✅' if status['isDataAvailable'] else 'Waiting...'}")
            
            if progress:
                print(f"   Input rows: {progress.get('numInputRows', 0)}")
                print(f"   Processing rate: {progress.get('processedRowsPerSecond', 0):.1f} rows/sec")
        
        time.sleep(30)
    
    print("\n✅ Monitoring terminé")

# Décommenter pour monitorer
# monitor_streams(5)

---
## Vérification des données écrites

In [ ]:
# Attendre quelques batches
import time
print("⏳ Attente de 2 minutes pour accumulation de données...")
time.sleep(120)

# Lire les données écrites
df_written = spark.read.parquet(OUTPUT_PATH)

print(f"\n📊 Données dans Data Lake:")
print(f"   Total transactions: {df_written.count():,}")

# Distribution par risk level
print("\n📈 Distribution par niveau de risque:")
df_written.groupBy("riskLevel").count().orderBy("count", ascending=False).show()

In [ ]:
# Statistiques détaillées
print("📊 Statistiques:")
df_written.select(
    F.count("*").alias("total_tx"),
    F.round(F.sum("amount"), 2).alias("total_volume"),
    F.round(F.avg("amount"), 2).alias("avg_amount"),
    F.round(F.avg("fraudScore"), 4).alias("avg_fraud_score"),
    F.sum(F.when(F.col("isFraud"), 1).otherwise(0)).alias("actual_frauds"),
    F.sum(F.when(F.col("isFraudPredicted"), 1).otherwise(0)).alias("predicted_frauds")
).show()

---
## Arrêter les streams

In [ ]:
# Arrêter tous les streams actifs
for query in spark.streams.active:
    print(f"Arrêt de: {query.id}")
    query.stop()

print("\n✅ Tous les streams arrêtés")

---
## Résumé

### Ce que nous avons fait:

1. ✅ **Lecture Event Hubs** - Streaming temps réel depuis Kafka-compatible
2. ✅ **Parsing JSON** - Extraction des champs avec schéma typé
3. ✅ **Enrichissement** - Ajout de features (heure, jour, ratios)
4. ✅ **Scoring fraude** - Règles métier avec pondération
5. ✅ **Écriture Data Lake** - Parquet partitionné par risque et date
6. ✅ **Alertes** - Extraction et stockage des transactions à risque

### Prochaines étapes:
- Créer les tables Synapse SQL sur les données Parquet
- Entraîner un modèle ML avec MLlib
- Configurer Data Factory pour l'orchestration